# 02 - Classifier training and SHAP

XGBoost with SMOTE applied *inside* each CV fold, then a held-out evaluation and SHAP explanations. MLflow logging and registration live in `classifier/train.py`; this notebook is for interactive experiments.

> **Caveat:** labels are rule-derived (see `docs/labelling_methodology.md`), so near-perfect scores mostly show that the model recovered the rules.

In [ ]:
import sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))   # make the project packages importable from the notebook

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

from classifier.train import DEFAULT_PARAMS, cross_validate, load_labelled, make_synthetic_labelled_data, _smote_k
from classifier.evaluate import CLASS_NAMES, evaluate_model, plot_confusion_matrix, shap_values_for
from classifier.feature_engineering import FEATURE_COLUMNS

labelled = ROOT / "data" / "labelled" / "labelled_findings.csv"
X, y = load_labelled(labelled) if labelled.exists() else make_synthetic_labelled_data(1500)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_test.shape, y_train.value_counts().sort_index().to_dict())

## 5-fold cross-validation (SMOTE per fold)

In [ ]:
cv = cross_validate(X_train, y_train, DEFAULT_PARAMS, n_splits=5)
print(f"macro-F1 = {cv['mean']:.4f} +/- {cv['std']:.4f}")
pd.Series(cv["fold_scores"], index=[f"fold {i}" for i in range(1, 6)]).round(4)

## Final model and held-out metrics

In [ ]:
X_res, y_res = SMOTE(k_neighbors=_smote_k(y_train), random_state=42).fit_resample(X_train, y_train)
model = XGBClassifier(num_class=len(CLASS_NAMES), **DEFAULT_PARAMS).fit(X_res, y_res)
metrics = evaluate_model(model, X_test, y_test)
print(f"macro-F1 {metrics['macro_f1']:.4f}   accuracy {metrics['accuracy']:.4f}")
pd.DataFrame(metrics["report"]).T.round(3)

In [ ]:
import tempfile
from PIL import Image
path = plot_confusion_matrix(y_test, model.predict(X_test), Path(tempfile.gettempdir()) / "cm.png")
plt.figure(figsize=(6, 5)); plt.imshow(Image.open(path)); plt.axis("off"); plt.show()

## SHAP: which features drive each class?

In [ ]:
import numpy as np
values = shap_values_for(model, X_test)            # (samples, features, classes)
importance = pd.DataFrame(np.abs(values).mean(axis=0), index=FEATURE_COLUMNS, columns=CLASS_NAMES)
importance.round(3).sort_values("REMEDIATE_NOW", ascending=False)

In [ ]:
ax = importance.plot.barh(stacked=True, figsize=(7, 3.5), colormap="Blues")
ax.set_xlabel("mean |SHAP value|"); ax.set_title("Feature importance by class")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()